# Scherrer, Lynch et al. (2023), dandiset 000691: reduced runs of the decoder-size and map analyses

Paper. Scherrer JR, Lynch GF, Zhang JJ, Fee MS (2023). An optical design enabling lightweight
and large field-of-view head-mounted microscopes. *Nature Methods* 20, 546-549.
[doi:10.1038/s41592-023-01806-1](https://doi.org/10.1038/s41592-023-01806-1).

Claims. `claims/000691/10.1038-s41592-023-01806-1.yaml`, written from the full text of the
version of record and the paper's code (GitHub `FeeLab/Feescope-Paper-Code`, commit d596646). It
decides what is tested and how.

Dandiset. [DANDI:000691](https://dandiarchive.org/dandiset/000691/0.260827.1358), version
0.260827.1358, the only published version (the paper cites none); see the main notebook,
`reproduce_scherrer_2023.ipynb`, for the reasons and the full data check.

Why this notebook. fig2k-decoder-size, fig2l-local-decoding-map and ed9c-local-decoding-map need
64,000 to 276,000 fits each, 4 to 11 hours on four cores, and are blocked as not runnable (their
cost is measured in the decoding notebooks). The first verification asked for any output of
them. This notebook runs each with far fewer fits, subsets, grid points and null decoders, choices
made here to fit the time limit of a notebook. They are not the criteria's methods and decide no
outcome; they show what the analyses give at this size.

| Claim | Data | Status |
|---|---|---|
| fig2k-decoder-size | dandiset | reduced run (16 subsets of 5 fits per size, in place of 64 of 100) |
| fig2l-local-decoding-map | dandiset | reduced run (grid every 50 pixels, 500 null decoders of one fit) |
| ed9c-local-decoding-map | dandiset | reduced run (as fig2l, 10 fits per point in place of 40) |

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

import kiloscope as ks

pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)
started = time.time()
urls = ks.asset_urls()
f = ks.open_nwb(urls)
S = ks.load_session(f)
POS = ks.mask_positions(ks.read_masks(urls[ks.NWB_PATH], f))
track, B = ks.load_behaviour(urls, S)
run = B["running"]
X_run = ks.moving_max(S["traces"][B["sel"]])[run]
X_F = np.asfortranarray(X_run)         # columns contiguous, for fast shuffles
print(f"loaded and tracked in {time.time() - started:.0f} s")

## Data check

The full check of the data is in the main notebook. Here: what this notebook decodes.

In [ ]:
LABELS_QD = ks.quad_dir_labels(B)[run]
LABELS_OCT = B["octant"][run]
print(f"imaging frames within the tracked video: {len(B['sel'])}; running (speed above its median): {run.sum()}")
print(f"activity: moving maximum over 30 frames of {X_run.shape[1]} footprints; values {X_run.min():.3f} to {X_run.max():.3f}")
print("frames per quadrant and direction class:", np.bincount(LABELS_QD, minlength=8).tolist())
print("frames per octant:", np.bincount(LABELS_OCT, minlength=8).tolist())
print(f"footprints with a position (the code's peak pixel): {(~np.isnan(POS['peak']).any(1)).sum()} of {len(POS['peak'])}")

## fig2k-decoder-size

> Decoding accuracy depends on the number of footprints used, with P at or below 2.7 x 10^-4
> (Mann-Whitney U) for every decoder size from 2 to 2,048 footprints.

Criterion: quadrant and direction labels, lambda 6.75e-8; for each size 2^1 to 2^11, 64 random
subsets, each the average matrix of 100 fits and the same subset shuffled, again 100 fits;
two-sided Mann-Whitney U under (a) original against shuffled at each size, (b) original at each
size against the next size. Here: 16 subsets of 5 fits each, original and shuffled.

In [ ]:
rng_k = np.random.default_rng(691_41)
t0 = time.time()
rows, orig, shuf = [], {}, {}
for size in 2 ** np.arange(1, 12):
    o_, s_ = [], []
    for _ in range(16):
        cols = rng_k.choice(X_run.shape[1], size, replace=False)
        o_.append(ks.decoding_matrix(X_run[:, cols], LABELS_QD, 8, ks.LAMBDA_OTHER, 5, rng_k, n_jobs=4)["median"])
        s_.append(ks.decoding_matrix(ks.shuffle_columns(X_F[:, cols], rng_k), LABELS_QD, 8, ks.LAMBDA_OTHER, 5, rng_k, n_jobs=4)["median"])
    orig[size], shuf[size] = np.array(o_), np.array(s_)
sizes = sorted(orig)
for i, size in enumerate(sizes):
    pa = mannwhitneyu(orig[size], shuf[size], alternative="two-sided").pvalue
    pb = mannwhitneyu(orig[size], orig[sizes[i + 1]], alternative="two-sided").pvalue if i + 1 < len(sizes) else np.nan
    rows.append({"size": size, "median_original": round(np.median(orig[size]), 3), "median_shuffled": round(np.median(shuf[size]), 3),
                 "NaN_original": int(np.isnan(orig[size]).sum()), "NaN_shuffled": int(np.isnan(shuf[size]).sum()),
                 "p_(a)_vs_shuffled": pa, "p_(b)_vs_next_size": pb})
tab_k = pd.DataFrame(rows)
print(f"{len(sizes)} sizes x 16 subsets x 2 x 5 fits in {time.time() - t0:.0f} s")
print(tab_k.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
for col, p_ in [("p_(a)_vs_shuffled", tab_k["p_(a)_vs_shuffled"]), ("p_(b)_vs_next_size", tab_k["p_(b)_vs_next_size"][:-1])]:
    print(f"{col}: every p <= 2.7e-4: {bool((p_ <= 2.7e-4).all())}; every p < 0.05: {bool((p_ < 0.05).all())}; "
          f"largest {np.nanmax(p_):.3g}; undefined (NaN, all values tied) {int(p_.isna().sum())}")
print(f"median original accuracy at 2,048 above that at 2: {np.median(orig[2048]) > np.median(orig[2])}")

**Result: blocked (not_runnable).** In this reduced run (16 subsets of 5 fits per size) the median
accuracy rises from 0.099 at 2 footprints to 0.442 at 2,048, and the original subsets differ from
the shuffled ones with p of 1.5e-6 to 6.7e-6 from 32 footprints on, but not at 2 to 8 footprints
(p 0.53 to 0.87) nor, at the 2.7e-4 level, at 16 (p 0.017). Neither reading has every p at or below
2.7e-4. This run is far smaller than the criterion's 64 subsets of 100 fits and decides nothing.

## fig2l-local-decoding-map

> Decoders built from the 32 footprints closest to a point decode quadrant and direction with
> accuracies that differ substantially across the field of view.

Criterion: map points every 5 pixels, a point used when at least 32 footprints lie within 270 um;
at each, the average matrix of 10 fits on its 32 nearest footprints; the null is 1,000 decoders on
shuffled data from 32 random footprints; Benjamini-Hochberg 5%; reproduced when at least one point
is significant, significant and non-significant points are both present (i), and the values span
at least 0.108 (ii), under every combination of readings. Here one combination, reduced: every
tenth grid point along each axis (every 50 pixels), the code's radius of 50 pixels and its
position, footprints drawn anew for each of 500 null decoders of one fit, both definitions of p.

In [ ]:
def reduced_map(labels, n_fits, rng, null_seed, n_null=500, every=10, radius=50):
    grid = ks.map_grid(every)
    t0 = time.time()
    used, vals = ks.map_values(X_run, labels, 8, ks.LAMBDA_OTHER, POS["peak"], grid, radius, 32, n_fits, rng)
    t_map = time.time() - t0
    t0 = time.time()
    null = ks.null_local(X_run, labels, 8, ks.LAMBDA_OTHER, n_null, 32, 1, seed=null_seed)
    print(f"grid of {len(grid)} points (every {every * 5} pixels), {len(used)} used (at least 32 footprints within {radius} pixels); "
          f"{n_fits} fits per point in {t_map:.0f} s; {n_null} null decoders of one fit in {time.time() - t0:.0f} s")
    print(f"point values: percentiles 0/5/50/95/100 {np.round(np.percentile(vals, [0, 5, 50, 95, 100]), 3)}; span {vals.max() - vals.min():.3f}")
    print(f"null values: percentiles 0/50/95/99/100 {np.round(np.nanpercentile(null, [0, 50, 95, 99, 100]), 3)}; NaN {np.isnan(null).sum()}")
    k = np.array([(null >= v).sum() for v in vals])
    rows = []
    for p_label, p in [("fraction", k / n_null), ("(k + 1) / (n + 1)", (k + 1) / (n_null + 1))]:
        sig = ks.benjamini_hochberg(p, 0.05)
        rows.append({"p": p_label, "significant": int(sig.sum()), "not_significant": int((~sig).sum()),
                     "both_present_(i)": bool(sig.any() and (~sig).any())})
    print(pd.DataFrame(rows).to_string(index=False))
    return used, vals, null

used_l, vals_l, null_l = reduced_map(LABELS_QD, 10, np.random.default_rng(691_42), null_seed=691_43)
print(f"(ii) span of the used points' values at least 0.108: {vals_l.max() - vals_l.min() >= 0.108}")

**Result: blocked (not_runnable).** In this reduced run (56 points used on a grid every 50 pixels,
500 null decoders of one fit) the point values span 0.094 to 0.281 (0.187), and 32 of 56 points are
significant with p as a fraction (31 with (k + 1) / (n + 1)), so (i) and (ii) hold for this one
combination of readings. It is far smaller than the criterion's run and decides nothing.

## ed9c-local-decoding-map

> Decoders built from the 32 footprints closest to a point decode octant position significantly in
> parts of the field of view, with accuracy that varies across it.

Criterion: as fig2l with octant labels and the average matrix of 40 fits per point, judged on (i);
the span is reported. Here as the reduced fig2l run, with 10 fits per point.

In [ ]:
used_o, vals_o, null_o = reduced_map(LABELS_OCT, 10, np.random.default_rng(691_44), null_seed=691_45)

**Result: blocked (not_runnable).** In this reduced run (56 points, 10 fits per point in place of
40, 500 null decoders of one fit) the point values span 0.078 to 0.203 (0.125), and 5 of 56 points
are significant under both definitions of p, so (i) holds for this one combination of readings.
It is far smaller than the criterion's run and decides nothing.

In [ ]:
print(f"total run time {(time.time() - started) / 60:.1f} min")

## Summary

| Claim | Outcome (this attempt) |
|---|---|
| fig2k-decoder-size | blocked: not runnable |
| fig2l-local-decoding-map | blocked: not runnable |
| ed9c-local-decoding-map | blocked: not runnable |

How the analysis differs from the paper's. Besides the differences of the decoding notebooks (all
2948 footprints, position tracked from the video, scikit-learn's solvers), every run here is
reduced in size by a choice made here, so none decides an outcome.